# 4. Download OpenTargets and source snapshots

Purpose: discover, download, or verify raw OpenTargets/source snapshots separately from transformation into KG nodes/edges.

Prerequisites:
- Notebook 3 has established access and selected `OPEN_TARGETS_RELEASE`.
- Writable local/scratch source root, defaulting to `data/opentargets/`.
- Network is optional and disabled by default.

Outputs:
- Raw OpenTargets Parquet directories under `data/opentargets/{dataset}/` or `JOUVENCE_OT_ROOT` (legacy fallback: `TXGNN_OT_ROOT`).
- A small build manifest under `.omoc/notebook-manifests/opentargets_sources.json` when writes are enabled.
- Dataset status table: path, shard count, schema sample, and completion marker.

Expected runtime:
- Default dry run: seconds; no downloads.
- Full run: minutes to hours depending on datasets and network.

Safe sample mode:
- `DRY_RUN=True` unless `JOUVENCE_NOTEBOOK_ALLOW_NETWORK=1` and `JOUVENCE_NOTEBOOK_ALLOW_WRITES=1` are both set (legacy fallbacks: `TXGNN_NOTEBOOK_ALLOW_NETWORK` and `TXGNN_NOTEBOOK_ALLOW_WRITES`).
- Download cells call existing `txdata_download.download_opentargets_datasets`; they do not implement download logic inline.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    # Allows running the notebook from notebooks/ in Jupyter.
    candidate = REPO_ROOT.parent
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
sys.path.insert(0, str(REPO_ROOT))

SAMPLE_MODE = os.environ.get("JOUVENCE_NOTEBOOK_SAMPLE_MODE", os.environ.get("TXGNN_NOTEBOOK_SAMPLE_MODE", "1")) != "0"
ALLOW_NETWORK = os.environ.get("JOUVENCE_NOTEBOOK_ALLOW_NETWORK", os.environ.get("TXGNN_NOTEBOOK_ALLOW_NETWORK", "0")) == "1"
ALLOW_WRITES = os.environ.get("JOUVENCE_NOTEBOOK_ALLOW_WRITES", os.environ.get("TXGNN_NOTEBOOK_ALLOW_WRITES", "0")) == "1"
OPEN_TARGETS_RELEASE = os.environ.get("OPEN_TARGETS_RELEASE", "latest")
DEFAULT_LOCAL_CACHE_ROOT = Path(os.environ.get("JOUVENCE_VERIFIED_KG_ROOT", os.environ.get("TXGNN_VERIFIED_KG_ROOT", "/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")))
LOCAL_CACHE_ROOT = Path(os.environ.get("JOUVENCE_LOCAL_CACHE_ROOT", os.environ.get("TXGNN_LOCAL_CACHE_ROOT", DEFAULT_LOCAL_CACHE_ROOT)))
DATA_ROOT = Path(os.environ.get("JOUVENCE_DATA_ROOT", os.environ.get("TXGNN_DATA_ROOT", REPO_ROOT / "data")))
OT_ROOT = Path(os.environ.get("JOUVENCE_OT_ROOT", os.environ.get("TXGNN_OT_ROOT", DATA_ROOT / "opentargets")))
KG_ROOT_URI = os.environ.get("JOUVENCE_KG_ROOT", os.environ.get("TXGNN_KG_ROOT", str(DATA_ROOT / "kg")))
CANONICAL_GCS_ROOT = "gs://jouvencekb/kg/v2"
CANONICAL_MOUNT_ROOT = Path(os.environ.get("JOUVENCE_VERIFIED_KG_ROOT", os.environ.get("TXGNN_VERIFIED_KG_ROOT", "/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")))

print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "sample_mode": SAMPLE_MODE,
    "allow_network": ALLOW_NETWORK,
    "allow_writes": ALLOW_WRITES,
    "open_targets_release": OPEN_TARGETS_RELEASE,
    "local_cache_root": str(LOCAL_CACHE_ROOT),
    "ot_root": str(OT_ROOT),
    "kg_root_uri": KG_ROOT_URI,
}, indent=2))

DRY_RUN = SAMPLE_MODE or not (ALLOW_NETWORK and ALLOW_WRITES)
print({"dry_run": DRY_RUN})


## Source dataset tranche

This notebook starts with node/core setup datasets. Later edge/evidence notebooks expand to heavier evidence and interaction tranches.


In [ ]:
from txdata_download import download_opentargets_dataset, download_opentargets_datasets, list_opentargets_datasets, get_latest_opentargets_release

CORE_SOURCE_DATASETS = ["target", "disease", "drug_molecule", "biosample", "go", "reactome"]
EDGE_SOURCE_DATASETS_FOR_LATER = [
    "target_homologues", "interaction", "evidence", "drug_indication",
    "drug_mechanism_of_action", "target_essentiality", "disease_phenotype",
    "expression", "pharmacogenomics", "variant", "known_variant", "enhancer_to_gene",
]
SELECTED_DATASETS = os.environ.get("JOUVENCE_OT_DATASETS", os.environ.get("TXGNN_OT_DATASETS", ",".join(CORE_SOURCE_DATASETS))).split(",")
SELECTED_DATASETS = [d.strip() for d in SELECTED_DATASETS if d.strip()]
release = get_latest_opentargets_release() if (OPEN_TARGETS_RELEASE == "latest" and ALLOW_NETWORK) else OPEN_TARGETS_RELEASE
print({"release": release, "selected_datasets": SELECTED_DATASETS, "ot_root": str(OT_ROOT)})


## Download or verify source snapshots

The full path below delegates to the existing downloader. In dry run, the cell prints the exact command and inspects whatever is already cached locally.


In [ ]:
if DRY_RUN:
    print("DRY_RUN: not downloading. To run the core tranche manually:")
    print(
        "uv run python - <<'PY'\n"
        "from pathlib import Path\n"
        "from txdata_download import download_opentargets_datasets\n"
        f"download_opentargets_datasets({SELECTED_DATASETS!r}, Path({str(OT_ROOT)!r}), release={release!r}, workers=4)\n"
        "PY"
    )
    downloaded = {}
else:
    downloaded = download_opentargets_datasets(
        datasets=SELECTED_DATASETS,
        dest_dir=OT_ROOT,
        release=release,
        workers=int(os.environ.get("JOUVENCE_OT_DOWNLOAD_WORKERS", os.environ.get("TXGNN_OT_DOWNLOAD_WORKERS", "4"))),
    )
downloaded


## Snapshot status and schema samples

Validation reads at most the first Parquet shard per dataset and counts shards; it never loads complete datasets into pandas.


In [ ]:
def parquet_files(path: Path) -> list[Path]:
    return sorted(path.glob("*.parquet")) if path.exists() else []


def first_parquet_schema(path: Path) -> dict:
    files = parquet_files(path) if path.is_dir() else ([path] if path.exists() else [])
    if not files:
        return {"exists": False, "path": str(path), "parquet_files": 0}
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(files[0])
    return {
        "exists": True,
        "path": str(files[0]),
        "parquet_files": len(files),
        "rows_in_first_file": pf.metadata.num_rows,
        "columns": pf.schema_arrow.names,
    }


def duckdb_count(path: Path) -> int | None:
    if not path.exists():
        return None
    import duckdb
    target = str(path / "*.parquet") if path.is_dir() else str(path)
    with duckdb.connect() as con:
        return con.execute("select count(*) from read_parquet(?)", [target]).fetchone()[0]

status_rows = []
for dataset in SELECTED_DATASETS:
    path = OT_ROOT / dataset
    files = parquet_files(path)
    marker = path / ".ot_complete"
    schema = first_parquet_schema(path)
    status_rows.append({
        "dataset": dataset,
        "path": str(path),
        "exists": path.exists(),
        "parquet_files": len(files),
        "complete_marker": marker.exists(),
        "sample_columns": schema.get("columns", [])[:12],
        "rows_in_first_file": schema.get("rows_in_first_file"),
    })
status = pd.DataFrame(status_rows)
status


## Build manifest

Writes are gated; the manifest records paths and release strings but no credentials.


In [ ]:
manifest = {
    "release": release,
    "source_root": str(OT_ROOT),
    "datasets": status.to_dict(orient="records"),
    "downloaded_this_run": {k: str(v) for k, v in downloaded.items()} if isinstance(downloaded, dict) else {},
    "dry_run": DRY_RUN,
}
manifest_path = Path(os.environ.get("JOUVENCE_NOTEBOOK_MANIFEST_PATH", os.environ.get("TXGNN_NOTEBOOK_MANIFEST_PATH", REPO_ROOT / "artifacts" / "cache" / "t_fc4ede1f" / "notebook-manifests" / "opentargets_sources.json")))
if ALLOW_WRITES:
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
    print(f"Wrote manifest: {manifest_path}")
else:
    print(f"ALLOW_WRITES=0; manifest not written. Intended path: {manifest_path}")
manifest


## Validation

In dry run, validation confirms the notebook is configured and reports missing cached data as actionable status. In full run, every selected dataset must have Parquet shards.


In [ ]:
assert SELECTED_DATASETS, "at least one dataset must be selected"
assert str(OT_ROOT), "OT_ROOT must be configured"
if not DRY_RUN:
    missing = status.loc[status["parquet_files"] == 0, "dataset"].tolist()
    assert not missing, f"Full download expected Parquet shards for: {missing}"
assert not any("token" in str(value).lower() or "password" in str(value).lower() for value in manifest.values())
print("Notebook 4 lightweight validation passed.")
